# WEAR-VQA gaze-grounded eval (SmolVLM2)

Robust eval on your **`wearvqa_gaze_only`** dataset (2 samples × 10 question types
= 20 egocentric VQA examples, each with **ground-truth gaze**).

Pipeline per sample: rater -> visual (all layers) -> **sink removal** (B baseline
subtraction + A drop invariant sinks). Then we compare the model's image-token
importance against the **human gaze point** — both visually (heatmap + gaze marker)
and quantitatively (peak→gaze distance and gaze-in-top-k, raw vs debiased vs random).

> **Runtime:** GPU runtime. Needs the dataset in your Google Drive at
> `MyDrive/wearvqa_gaze_only`.

## 1. Mount Drive + install + clone

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words pytest matplotlib
!rm -rf text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git
%cd text_vision_attention_map

## 2. Setup + sample 2 per type
Set `DATA_ROOT` if your folder lives elsewhere in Drive.

In [ ]:
import importlib.util, os, math, glob, json
import numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

DATA_ROOT = "/content/drive/MyDrive/wearvqa_gaze_only"
N_PER_TYPE = 2
assert os.path.isdir(DATA_ROOT), f"not found: {DATA_ROOT}  (fix DATA_ROOT)"

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS
from transformers import AutoProcessor
tokenizer = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-2.2B-Instruct").tokenizer

TYPES = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
samples = []
for t in TYPES:
    for jp in sorted(glob.glob(os.path.join(DATA_ROOT, t, "*.json")))[:N_PER_TYPE]:
        meta = json.load(open(jp))
        img_path = jp[:-5] + ".jpg"
        if not os.path.exists(img_path) or "gaze" not in meta:
            continue
        samples.append(dict(type=t, img_path=img_path, question=meta["question"],
                            gaze=meta["gaze"], rationale=meta["gaze"].get("rationale", "")))
print(f"types: {len(TYPES)}  |  sampled: {len(samples)} examples")

def to_grid(vec, L_v):
    g = int(round(math.sqrt(L_v)))
    if g * g != L_v:
        g = math.ceil(math.sqrt(L_v)); vec = np.concatenate([vec, np.zeros(g*g-L_v, vec.dtype)])
    return vec.reshape(g, g)

def up(a, size, mode=Image.BILINEAR):
    a = (a / (a.max() + 1e-9) * 255).astype('uint8')
    return np.array(Image.fromarray(a).resize(size, mode))

def mark_gaze(ax, gz, size):
    W, H = size
    ax.scatter([gz["x_norm"] * W], [gz["y_norm"] * H], marker="x", s=150, c="lime", linewidths=3)

## 3. Run the pipeline on all samples, build the global sink baseline, debias
First run downloads the ~4.5 GB weights.

In [ ]:
def importance_for(s):
    img = S.load_image(s["img_path"])
    o = S.make_smolvlm_output(image=img, question=s["question"])
    if o is None:
        return None
    maps, tpos, _ = RS.sliced_maps_from_full(o.raw_scores, o.image_token_mask, o.text_token_mask)
    tt = tokenizer.convert_ids_to_tokens(o.input_ids[tpos].tolist())
    rr = RS.select_important_text_tokens(maps, text_tokens=tt, tokenizer=tokenizer,
                                         question=s["question"], pct=0.5)
    imp, _, _, _ = VS.image_importance(maps, rr.rater_mask)
    s.update(img=img, raters=rr.kept_tokens(tt), importance=imp)
    return s

data = [importance_for(s) for s in samples]
data = [d for d in data if d is not None and "importance" in d]
L_v = data[0]["importance"].numel()
assert all(d["importance"].numel() == L_v for d in data), "L_v differs across samples"

# --- baseline: LEAVE-ONE-OUT, so an example never corrects itself -------------
# A plain make_baseline([all]) would give each example 1/N of its own correction,
# making every map depend on which OTHER examples were sampled. loo[i] is the mean
# over every example except i. (Best available here without extra model passes;
# a frozen held-out or per-image null-prompt baseline is stronger still.)
loo = VS.make_baseline_loo([d["importance"] for d in data])   # [N, L_v]
baseline_all = VS.make_baseline([d["importance"] for d in data])   # reporting only
sink = int(baseline_all.argmax())

DROP_SINK_K = 3
for i, d in enumerate(data):
    d["baseline"] = loo[i]
    # selection path (top-k heatmaps for section 5) -- clamping is fine here
    deb = VS.subtract_baseline(d["importance"], loo[i])
    deb = deb.clone(); deb[VS.sink_token_mask(loo[i], DROP_SINK_K)] = 0.0
    ssum = deb.sum(); d["debiased"] = deb / ssum if ssum > 0 else deb
    # label path -- log-space (PMI), sinks EXCLUDED from cand rather than zeroed
    d["label"] = VS.teacher_label(d["importance"], loo[i], drop_sink_k=DROP_SINK_K)

lab0 = data[0]["label"]
print(f"{len(data)} samples | L_v={L_v} | global sink patch={sink}")
print(f"labels: {lab0.n_cand}/{lab0.L_v} candidates | "
      f"teacher dist min={lab0.distribution().min():.2e} (dense, no zero-support)")

## 4. Quantitative: does the selection match the human gaze?
Map gaze (`x_norm`,`y_norm`) to a patch on the L_v grid, then compare the model's
peak to it. Lower distance / higher hit-rate = better gaze alignment.

In [ ]:
G = int(round(math.sqrt(L_v)))
def rc(idx):        return idx // G, idx % G
def gaze_patch(gz): return min(G-1, int(gz["y_norm"]*G)) * G + min(G-1, int(gz["x_norm"]*G))
def gdist(idx, gp): (r1,c1),(r2,c2) = rc(idx), rc(gp); return ((r1-r2)**2 + (c1-c2)**2) ** 0.5

K = 9   # top-k for hit-rate (=pct 0.9)
for d in data:
    gp = gaze_patch(d["gaze"]); d["gaze_patch"] = gp
    d["dist_raw"] = gdist(int(d["importance"].argmax()), gp)
    d["dist_deb"] = gdist(int(d["debiased"].argmax()), gp)
    d["dist_rand"] = float(np.mean([gdist(p, gp) for p in range(L_v)]))
    d["hit_raw"] = gp in set(torch.topk(d["importance"], K).indices.tolist())
    d["hit_deb"] = gp in set(torch.topk(d["debiased"], K).indices.tolist())

def mean(k): return float(np.mean([d[k] for d in data]))
print("peak -> gaze distance (grid patches, lower = better):")
print(f"   raw      : {mean('dist_raw'):.2f}")
print(f"   debiased : {mean('dist_deb'):.2f}")
print(f"   random   : {mean('dist_rand'):.2f}")
print(f"\ngaze-in-top{K} hit rate (higher = better):")
print(f"   raw      : {mean('hit_raw'):.0%}")
print(f"   debiased : {mean('hit_deb'):.0%}")
print(f"   random   : {K / L_v:.0%}")

print("\nper-type mean peak->gaze distance (raw -> debiased):")
by = defaultdict(list)
for d in data: by[d["type"]].append(d)
for t in sorted(by):
    dr = np.mean([x["dist_raw"] for x in by[t]]); dd = np.mean([x["dist_deb"] for x in by[t]])
    print(f"   {t:<38} {dr:4.2f} -> {dd:4.2f}")

## 5. Visual grid: image | raw importance | debiased  (green × = human gaze)

In [ ]:
fig, axes = plt.subplots(len(data), 3, figsize=(11, 3.3 * len(data)))
if len(data) == 1:
    axes = axes[None, :]
for r, d in enumerate(data):
    img = d["img"]; sz = img.size
    raw_h = to_grid(d["importance"].numpy(), L_v)
    deb_h = to_grid(d["debiased"].numpy(), L_v)
    axes[r, 0].imshow(img); mark_gaze(axes[r, 0], d["gaze"], sz); axes[r, 0].axis("off")
    axes[r, 0].set_title(f"[{d['type']}] {d['question']}\nraters: {d['raters']}  gaze: {d['rationale']}", fontsize=7)
    axes[r, 1].imshow(img); axes[r, 1].imshow(up(raw_h, sz), cmap="jet", alpha=0.5)
    mark_gaze(axes[r, 1], d["gaze"], sz); axes[r, 1].axis("off")
    axes[r, 1].set_title(f"raw (dist {d['dist_raw']:.1f})", fontsize=8)
    axes[r, 2].imshow(img); axes[r, 2].imshow(up(deb_h, sz), cmap="jet", alpha=0.5)
    mark_gaze(axes[r, 2], d["gaze"], sz); axes[r, 2].axis("off")
    axes[r, 2].set_title(f"debiased (dist {d['dist_deb']:.1f})", fontsize=8)
plt.tight_layout(); plt.show()